## Basically a Sales Brochure Generator 

In [2]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from my_easy_scraper import fetch_all_website_links, fetch_website_contents, fetch_website_links 
from openai import OpenAI

In [3]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [6]:
links = fetch_all_website_links("https://github.com")
links

Crawling pages: 100%|██████████| 1/1 [00:00<00:00,  1.01 pages/s]


['https://archiveprogram.github.com',
 'https://cli.github.com',
 'https://desktop.github.com',
 'https://docs.github.com',
 'https://docs.github.com/get-started/exploring-integrations/about-building-integrations',
 'https://docs.github.com/search-github/github-code-search/understanding-github-code-search-syntax',
 'https://docs.github.com/site-policy/github-terms/github-terms-of-service',
 'https://docs.github.com/site-policy/privacy-policies/github-privacy-statement',
 'https://github.blog',
 'https://github.blog/changelog',
 'https://github.careers',
 'https://github.com',
 'https://github.com/',
 'https://github.com/about',
 'https://github.com/about/diversity',
 'https://github.com/accelerator',
 'https://github.com/collections',
 'https://github.com/customer-stories',
 'https://github.com/customer-stories/duolingo',
 'https://github.com/customer-stories/figma',
 'https://github.com/customer-stories/mercado-libre',
 'https://github.com/customer-stories/mercedes-benz',
 'https://gi

In [9]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://github.com"))


Here is the list of links on the website https://github.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#start-of-content
/
/login
https://github.com/features/copilot
https://github.com/features/spark
https://github.com/features/models
https://github.com/mcp
https://github.com/features/actions
https://github.com/features/codespaces
https://github.com/features/issues
https://github.com/features/code-review
https://github.com/security/advanced-security
https://github.com/security/advanced-security/code-security
https://github.com/security/advanced-security/secret-protection
https://github.com/why-github
https://docs.github.com
https://github.blog
https://github.blog/changelog
https://github.com/marketplace
https://github.com/features
https://github.com/enterprise
https://github.com/team
https://github.

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

select_relevant_links("https://github.com")
    

{'links': [{'type': 'about page', 'url': 'https://github.com/about'},
  {'type': 'about page', 'url': 'https://github.com/why-github'},
  {'type': 'team page', 'url': 'https://github.com/team'},
  {'type': 'enterprise page', 'url': 'https://github.com/enterprise'},
  {'type': 'partners page', 'url': 'https://github.com/partners'},
  {'type': 'partners page', 'url': 'https://partner.github.com'},
  {'type': 'newsroom page', 'url': 'https://github.com/newsroom'},
  {'type': 'roadmap page', 'url': 'https://github.com/github/roadmap'},
  {'type': 'careers page', 'url': 'https://github.careers'}]}

In [10]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

headers = {
    "Authorization": f"Bearer {api_key}"
}

# Get usage for current month
response = requests.get(
    "https://api.openai.com/v1/dashboard/billing/usage",
    headers=headers,
    params={
        "start_date": "2026-02-01",
        "end_date": "2026-04-14"
    }
)

usage_data = response.json()
print("Current usage (USD):", usage_data.get("total_usage", 0) / 100)

Current usage (USD): 0.0
